# DISCoVeR with Simulation Data

## Import Packages

In [ ]:
# import packages
import scanpy as sc
import torch, pyro
import torch.utils.data as utils
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import pyro.optim as opt
import seaborn as sns

from umap import UMAP

import sys
sys.path.append("../../src/discover/")

from NBVAE_variants import ZINBDLVAE
from VAE_trainers import AdversarialThresholdPyroTrainer
from matplotlib.colors import LinearSegmentedColormap, ListedColormap

# set seeds
np.random.seed(42)
torch.manual_seed(42)
pyro.util.set_rng_seed(42)

# define color maps
cmap_trt = LinearSegmentedColormap.from_list("cmap", ["#42378C", "#D9A404"])
cmap_ct = ListedColormap(sns.color_palette('colorblind').as_hex())

## Helper Functions

In [ ]:
def force_aspect(ax, aspect=1):
    """
    Force aspect ratio of a matplotlib axis to be equal. Helper function for plotting.
    Taken from Patches tutorial docs.

    Parameters
    ----------
    ax : matplotlib axis
        The axis to set the aspect ratio for.
    aspect : float, optional
        The aspect ratio to set. Default is 1 (equal).
    """
    im = ax.get_images()
    extent =  im[0].get_extent()
    ax.set_aspect(abs((extent[1]-extent[0])/(extent[3]-extent[2]))/aspect)

def create_umap_df(preds, adata, model):
    """
    Create a dataframe with UMAP and PCA reductions for plotting.
    Adapted from Patches tutorial docs.

    Parameters
    ----------
    preds : dict
        The model predictions.
    adata : anndata.AnnData
        The AnnData object.
    model : str
        The model type, either "Discover" or "Base".
    
    Returns
    -------
    pd.DataFrame
        A dataframe with UMAP and PCA reductions for plotting.
    """
    # UMAP reducers
    reducer_base = UMAP(n_neighbors=50, min_dist=0.1, metric="correlation", verbose=False, random_state=42)
    reducer = UMAP(n_neighbors=50, min_dist=0.1, metric="correlation", verbose=False, random_state=42)

    match model:
        case "Discover":
            # all data reductions needed for the plots
            base_umap = reducer_base.fit_transform(np.array(adata.X))
            z_umap = reducer.fit_transform(preds["z"][0].cpu()) # Z
            w_umap = reducer.fit_transform(preds["w"][0].cpu()) # W

            w_pca = sc.pp.pca(preds["w"][0].cpu().numpy(), random_state=42)[:,:2] # W PCA, grab first 2 PCs
            z_pca = sc.pp.pca(preds["z"][0].cpu().numpy(), random_state=42)[:,:2] # Z PCA, grab first 2 PCs

            # create dataframe and add all reductions
            df = pd.DataFrame(base_umap)
            df.index = adata.obs.index
            
            df.columns = ["base_1", "base_2"]
            df["z_umap_1"], df["z_umap_2"] = z_umap[:,0], z_umap[:,1]
            df["w_umap_1"], df["w_umap_2"] = w_umap[:,0], w_umap[:,1]
            df["z_pc_1"], df["z_pc_2"] = z_pca[:,0], z_pca[:,1]
            df["w_pc_1"], df["w_pc_2"] = w_pca[:,0], w_pca[:,1]

        case "Base":
            anndata = adata.copy()
            anndata.X = anndata.layers["counts"]
            
            sc.pp.normalize_total(anndata, target_sum=1e4)
            sc.pp.log1p(anndata)
            sc.tl.pca(anndata, svd_solver="arpack")

            base_umap = reducer_base.fit_transform(np.array(anndata.X))
            base_pca = anndata.obsm['X_pca'][:,:2]

            df = pd.DataFrame(base_umap)
            df.index = anndata.obs.index

            df.columns = ["base_umap_1", "base_umap_2"]
            df["base_pc_1"], df["base_pc_2"] = base_pca[:,0], base_pca[:,1]

    # add metadata
    df["group_id"], df["cluster_id"], df["sample_id"] = adata.obs["group_id"], adata.obs["cluster_id"], adata.obs["sample_id"]

    return df

## Load and Prepare Data

In [ ]:
# load data
adata = sc.read_h5ad("../../data/sim/01-pro/t100,s80,b0.h5ad")

# full adata
adata_full = adata.copy()

# highly variable gene selection on log-normalized data
adata.X = adata.layers["logcounts"]
sc.pp.highly_variable_genes(adata, n_top_genes=1500)
sc.pl.highly_variable_genes(adata)
adata_hvg = adata[:, adata.var["highly_variable"]].copy()

# reset to raw counts for model input (as stated in docs)
adata_full.X = adata_full.layers["counts"]
adata_hvg.X = adata_hvg.layers["counts"]

print(f"Full data shape: {adata_full.shape}, HVG data shape: {adata_hvg.shape}")

In [ ]:
adata_hvg

In [ ]:
np.random.seed(42)
torch.manual_seed(42)
pyro.util.set_rng_seed(42)

batch_size=128
x = torch.FloatTensor(adata_hvg.layers['counts'].copy())
y = torch.FloatTensor(adata_hvg.obs['group_id'].cat.codes.to_numpy().reshape(-1,1).copy())
y_info = torch.FloatTensor(adata_hvg.obs['cluster_id'].cat.codes.to_numpy().reshape(-1,1).copy())

dataset = utils.TensorDataset(x, y, y_info)
train_set, test_set = dataset, dataset
train_set, test_set = utils.TensorDataset(*train_set[:]), utils.TensorDataset(*test_set[:])
train_loader, test_loader  = torch.utils.data.DataLoader(train_set, shuffle=True, batch_size=batch_size),  torch.utils.data.DataLoader(test_set, shuffle=True, batch_size=batch_size)

## DISCoVeR

In [ ]:
np.random.seed(42)
torch.manual_seed(42)
pyro.util.set_rng_seed(42)

dlvae = ZINBDLVAE(1500, [1], latent_dim=10, w_dim=10, num_layers=0, hidden_dim=128, recon_weight=9e-1, recon_weight_z=1e-1, w_kl_weight=1e-4, z_kl_weight=1e-4, adversarial_weight=1e2, learnable_prior=False)
dlvae_trainer = AdversarialThresholdPyroTrainer(0, 50, 1, 1, dlvae, train_loader, test_loader, opt.AdamW({"lr": 1e-3}))
dlvae_trainer.train()

In [ ]:
preds = dlvae_trainer.get_variables('test')
z_s = preds['z'][0].cpu()
w_s = preds['w'][0].cpu()
recons_w = preds['rec_w'][0].cpu()
recons_z = preds['rec_z'][0].cpu()

In [ ]:
adata_hvg.obsm["z_discover"] = z_s.numpy()
adata_hvg.obsm["w_discover"] = w_s.numpy()

adata_hvg.write_h5ad("../../data/sim/04-emb/t100,s80,b0-discover_hvg.h5ad")

In [ ]:
trace = dlvae_trainer.get_trace('test')
print(-1 * trace.nodes['rec_w']['fn'].log_prob(test_set[:][0]).mean().item())
print(-1 * trace.nodes['rec_z']['fn'].log_prob(test_set[:][0]).mean().item())

plt.figure()
plt.scatter(test_set[:][0].log1p().mean(dim=0), recons_w.log1p().mean(dim=0))
plt.scatter(test_set[:][0].log1p().mean(dim=0), recons_z.log1p().mean(dim=0))

plt.gca().axis('square')

plt.figure()
plt.scatter(test_set[:][0].log1p().var(dim=0), recons_w.log1p().var(dim=0))
plt.scatter(test_set[:][0].log1p().var(dim=0), recons_z.log1p().var(dim=0))
plt.gca().axis('square')


plt.show()

In [ ]:
# create dataframes for plotting
df_discover = create_umap_df(preds, adata_hvg, "Discover")
df_base = create_umap_df(preds, adata_hvg, "Base")

In [ ]:
# Figure skeleton (adapted from Patches tutorial docs)


## color palettes
klee_palette = [
    "#8B1E3F",  # Deep Burgundy
    "#3B5998",  # Rich Blue
    "#F4A261",  # Warm Orange
    "#264653",  # Deep Teal
    "#E9C46A",  # Soft Yellow
    "#2A9D8F",  # Muted Green
    "#E76F51",  # Burnt Sienna
    "#D3D9E3",  # Soft Pastel Blue
    "#A8DADC",  # Pale Turquoise
    "#BC4749",  # Warm Cranberry Red
]

klee_palette_masch = [
    "#3B5998",  # Rich Blue
    "#6A994E",  # Fresh Olive Green
    "#F4A261",  # Warm Orange
    "#E9C46A",  # Soft Yellow
    "#2A9D8F",  # Muted Green
    "#E76F51",  # Burnt Sienna
    "#FFC8A2",  # Soft Peach
    "#A8DADC",  # Pale Turquoise
    "#BC4749",  # Warm Cranberry Red
]

cbf_palette = [
    "#E69F00",  # Orange
    "#56B4E9",  # Sky Blue
    "#009E73",  # Bluish Green
    "#F0E442",  # Yellow
    "#0072B2",  # Blue
    "#D55E00",  # Vermillion
    "#CC79A7",  # Reddish Purple
    "#999999",  # Gray
    "#117733",  # Dark Green
    "#882255",  # Wine
    "#44AA99",  # Teal
    "#DDCC77",  # Sand
    "#88CCEE",  # Light Blue
    "#AA4499",  # Magenta
    "#332288",  # Navy
    "#E17C05",  # Burnt Orange
    "#DC267F",  # Pink
    "#648FFF",  # Vivid Blue
    "#785EF0",  # Violet
    "#FE6100",  # Bright Orange
]


## plot parameters
fontsize=14
alpha=0.3
s=10
s_pca=3

group_colours = [cbf_palette[i] for i in [0, 1]]
condition_colours = [cbf_palette[i] for i in [2, 4, 6]]
sample_colours = [cbf_palette[i] for i in [2, 3, 4, 5, 6, 7]]

## create a figure with a 2x2 grid of subplots
fig = plt.figure(figsize=(21, 21))

## define a GridSpec with a 2x2 layout
gs = gridspec.GridSpec(2, 2, wspace=0.17, hspace = 0.3, figure=fig)

## create subplots for the 2x2 grid
ax = [fig.add_subplot(gs[i//2, i%2]) for i in range(4)]

for subax in ax:
    subax.axis('off')

## define a new GridSpec for axis to split vertically
gs_inner_topleft = gridspec.GridSpecFromSubplotSpec(2, 2, subplot_spec=gs[0, 0], wspace=0.1, hspace=0.15)
gs_inner_topright = gridspec.GridSpecFromSubplotSpec(2, 2, subplot_spec=gs[0, 1], wspace=0.1, hspace=0.15)
gs_inner_botleft = gridspec.GridSpecFromSubplotSpec(2, 2, subplot_spec=gs[1, 0], wspace=0.1, hspace=0.15)
gs_inner_botright = gridspec.GridSpecFromSubplotSpec(2, 4, subplot_spec=gs[1, 1], wspace=0.25)

## create subplots for the inner grid
ax_inner_topleft = [fig.add_subplot(gs_inner_topleft[i//2, i%2]) for i in range(4)]
ax_inner_topright = [fig.add_subplot(gs_inner_topright[i//2, i%2]) for i in range(4)]
ax_inner_botleft = [fig.add_subplot(gs_inner_botleft[i//2, i%2]) for i in range(4)]

## specific for botright
ax_inner_botright = [fig.add_subplot(gs_inner_botright[0,0])]
ax_inner_botright = ax_inner_botright \
+ [
    fig.add_subplot(gs_inner_botright[0,1],sharey=ax_inner_botright[0]),
    fig.add_subplot(gs_inner_botright[0,2],sharey=ax_inner_botright[0]),
    fig.add_subplot(gs_inner_botright[0,3],sharey=ax_inner_botright[0]),
    fig.add_subplot(gs_inner_botright[1,0],sharey=ax_inner_botright[0]),
    fig.add_subplot(gs_inner_botright[1,1],sharey=ax_inner_botright[0]),
    fig.add_subplot(gs_inner_botright[1,2],sharey=ax_inner_botright[0]),
    fig.add_subplot(gs_inner_botright[1,3],sharey=ax_inner_botright[0]),
]


## UMAP plots

### counts
clu = sns.scatterplot(df_base, x='base_umap_1', y='base_umap_2', ax=ax_inner_topright[0], hue = 'group_id', palette=sns.color_palette(group_colours), s=s, alpha=alpha)
con = sns.scatterplot(df_base, x='base_umap_1', y='base_umap_2', ax=ax_inner_topleft[0], hue = 'cluster_id', palette=sns.color_palette(condition_colours), s=s, alpha=alpha)
sam = sns.scatterplot(df_base, x='base_umap_1', y='base_umap_2', ax=ax_inner_botleft[0], hue = 'sample_id', palette=sns.color_palette(sample_colours), s=s, alpha=alpha)

### cell identities (not available for linear decoder)
ax_inner_topright[1].text(0.5, 0.5, 'N/A', horizontalalignment='center', verticalalignment='center', fontsize=fontsize*2, color='grey', alpha=0.5)
ax_inner_topleft[1].text(0.5, 0.5, 'N/A', horizontalalignment='center', verticalalignment='center', fontsize=fontsize*2, color='grey', alpha=0.5)
ax_inner_botleft[1].text(0.5, 0.5, 'N/A', horizontalalignment='center', verticalalignment='center', fontsize=fontsize*2, color='grey', alpha=0.5)

### Zs
sns.scatterplot(df_discover, x='z_umap_1', y='z_umap_2', ax=ax_inner_topright[2], hue = 'group_id', palette=sns.color_palette(group_colours), s=s, alpha=alpha, legend=False)
sns.scatterplot(df_discover, x='z_umap_1', y='z_umap_2', ax=ax_inner_topleft[2], hue = 'cluster_id', palette=sns.color_palette(condition_colours), s=s, alpha=alpha, legend=False)
sns.scatterplot(df_discover, x='z_umap_1', y='z_umap_2', ax=ax_inner_botleft[2], hue = 'sample_id', palette=sns.color_palette(sample_colours), s=s, alpha=alpha, legend=False)

### Ws
sns.scatterplot(df_discover, x='w_umap_1', y='w_umap_2', ax=ax_inner_topright[3], hue = 'group_id', palette=sns.color_palette(group_colours), s=s, alpha=alpha, legend=False)
sns.scatterplot(df_discover, x='w_umap_1', y='w_umap_2', ax=ax_inner_topleft[3], hue = 'cluster_id', palette=sns.color_palette(condition_colours), s=s, alpha=alpha, legend=False)
sns.scatterplot(df_discover, x='w_umap_1', y='w_umap_2', ax=ax_inner_botleft[3], hue = 'sample_id', palette=sns.color_palette(sample_colours), s=s, alpha=alpha, legend=False)


## PCA plots

### cluster
sns.stripplot(df_discover, y = "z_pc_1", hue='cluster_id', zorder=1, alpha=alpha, s=s_pca, ax=ax_inner_botright[0], legend=False, palette=sns.color_palette(condition_colours))
#ax_inner_botright[0].axvline(zorder=2, color='black', linestyle = 'dashed')

sns.stripplot(df_discover, y = "z_pc_2", hue='cluster_id', zorder=1, alpha=alpha, s=s_pca, ax=ax_inner_botright[1], legend=False, palette=sns.color_palette(condition_colours))
#ax_inner_botright[1].axvline(zorder=2, color='black', linestyle = 'dashed')

sns.stripplot(df_discover, y = "w_pc_1", hue='cluster_id', zorder=1, alpha=alpha, s=s_pca, ax=ax_inner_botright[2], legend=False, palette=sns.color_palette(condition_colours))
#ax_inner_botright[2].axvline(zorder=2, color='black', linestyle = 'dashed')

sns.stripplot(df_discover, y = "w_pc_2", hue='cluster_id', zorder=1, alpha=alpha, s=s_pca, ax=ax_inner_botright[3], legend=False, palette=sns.color_palette(condition_colours))


### condition
sns.stripplot(df_discover, y = "z_pc_1", hue='group_id', zorder=1, alpha=alpha, s=s_pca, ax=ax_inner_botright[4], legend=False, palette=sns.color_palette(group_colours))
#ax_inner_botright[4].axvline(zorder=2, color='black', linestyle = 'dashed')

sns.stripplot(df_discover, y = "z_pc_2", hue='group_id', zorder=1, alpha=alpha, s=s_pca, ax=ax_inner_botright[5], legend=False, palette=sns.color_palette(group_colours))
#ax_inner_botright[5].axvline(zorder=2, color='black', linestyle = 'dashed')

sns.stripplot(df_discover, y = "w_pc_1", hue='group_id', zorder=1, alpha=alpha, s=s_pca, ax=ax_inner_botright[6], legend=False, palette=sns.color_palette(group_colours))
#ax_inner_botright[6].axvline(zorder=2, color='black', linestyle = 'dashed')

sns.stripplot(df_discover, y = "w_pc_2", hue='group_id', zorder=1, alpha=alpha, s=s_pca, ax=ax_inner_botright[7], legend=False, palette=sns.color_palette(group_colours))
#ax_inner_botright[7].axvline(zorder=2, color='black', linestyle = 'dashed')


## formatting

for subax in ax_inner_topright:
    subax.set_xticklabels([])
    subax.set_xticks([])
    subax.set_yticklabels([])
    subax.set_yticks([])
    subax.set_xlabel('UMAP 1', fontsize=fontsize*0.6)
    subax.set_ylabel('UMAP 2', fontsize=fontsize*0.6)
    try:
        force_aspect(subax)
    except:
        pass

for subax in ax_inner_topleft:
    subax.set_xticklabels([])
    subax.set_xticks([])
    subax.set_yticklabels([])
    subax.set_yticks([])
    subax.set_xlabel('UMAP 1', fontsize=fontsize*0.6)
    subax.set_ylabel('UMAP 2', fontsize=fontsize*0.6)
    try:
        force_aspect(subax)
    except:
        pass

for subax in ax_inner_botleft:
    subax.set_xticklabels([])
    subax.set_xticks([])
    subax.set_yticklabels([])
    subax.set_yticks([])
    subax.set_xlabel('UMAP 1', fontsize=fontsize*0.6)
    subax.set_ylabel('UMAP 2', fontsize=fontsize*0.6)
    try:
        force_aspect(subax)
    except:
        pass

for subax in ax_inner_botright:
    subax.set_xticklabels([])
    subax.set_xticks([])
    subax.set_xlabel('')
    subax.set_ylabel('')

ax_inner_botright[0].set_ylabel('Principal Score', fontsize=fontsize*0.6)
ax_inner_botright[4].set_ylabel('Principal Score', fontsize=fontsize*0.6)

clu_h, clu_l = clu.get_legend_handles_labels() 
con_h, con_l = con.get_legend_handles_labels()
sam_h, sam_l = sam.get_legend_handles_labels()

clu.legend([], frameon=False); con.legend([], frameon=False); sam.legend([], frameon=False)

ax_inner_topleft[0].set_title('Normalized Counts', fontsize=fontsize)
ax_inner_topleft[1].set_title('Cell Identity (ρ)', fontsize=fontsize)
ax_inner_topleft[2].set_title('Common (Z)', fontsize=fontsize)
ax_inner_topleft[3].set_title('Conditional (W)', fontsize=fontsize)

ax_inner_topright[0].set_title('Normalized Counts', fontsize=fontsize)
ax_inner_topright[1].set_title('Cell Identity (ρ)', fontsize=fontsize)
ax_inner_topright[2].set_title('Common (Z)', fontsize=fontsize)
ax_inner_topright[3].set_title('Conditional (W)', fontsize=fontsize)

ax_inner_botleft[0].set_title('Normalized Counts', fontsize=fontsize)
ax_inner_botleft[1].set_title('Cell Identity (ρ)', fontsize=fontsize)
ax_inner_botleft[2].set_title('Common (Z)', fontsize=fontsize)
ax_inner_botleft[3].set_title('Conditional (W)', fontsize=fontsize)

ax_inner_botright[0].set_title('Z - PC 1', fontsize=fontsize)
ax_inner_botright[1].set_title('Z - PC 2', fontsize=fontsize)
ax_inner_botright[2].set_title('W - PC 1', fontsize=fontsize)
ax_inner_botright[3].set_title('W - PC 2', fontsize=fontsize)
ax_inner_botright[4].set_title('Z - PC 1', fontsize=fontsize)
ax_inner_botright[5].set_title('Z - PC 2', fontsize=fontsize)
ax_inner_botright[6].set_title('W - PC 1', fontsize=fontsize)
ax_inner_botright[7].set_title('W - PC 2', fontsize=fontsize)


## set subplot titles
titles = ['Groups (Types)', 'Conditions (States)', 'Samples', '']

for a, t in zip(ax, titles):
    a.set_title(t, y=1.05, fontsize=fontsize*1.2)


## save and show figure
fig.suptitle("DISCoVeR - Conditions", fontsize=18, y=0.95)
plt.savefig("../../outs/sim/t100,s80,b0-discover_con_full.png", dpi=300, bbox_inches='tight')
plt.show()